# 三模型 Pass@n 精度对比

本 notebook 从保留的正式 run 中自动提取 Qwen3.5-35B、Qwen3.5-397B 和 DeepSeek-V4-Flash-DSpark 的精度，并绘制三子图。配色沿用 `mtp_csd_pass_at_n.png` 的蓝色系列。

为保证三个面板方法一致，只比较 Bare、Plain CSD、Dynamic CSD、Dynamic CSD + Entropy Gate，不绘制仅 Qwen 实验包含的 Auto。

> 注意：当前 397B run 使用 `temperature=1.0, presence_penalty=1.5`，与模型卡推荐的 thinking 配置不同，因此该面板属于诊断结果。DSpark AIME 会优先读取正在补跑的 pass@16；四方法未全部完成时自动回退到 static pass@4。

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path('/root/sglang-dspark-csd')
OUT_DIR = REPO / 'runs/final_figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)

QWEN35 = REPO / 'runs/mtp_csd/qwen35_accuracy_avg_314_c48_20260802'
QWEN397 = REPO / 'runs/mtp_csd/qwen35_397b_accuracy_avg_314_tp8_c48_20260808/results'
QWEN397_MATH_RATIO04 = REPO / 'runs/mtp_csd/qwen35_397b_math500_ratio04_final_314_tp8_c48_20260811'
QWEN397_MATH_RATIO04_SD = QWEN397_MATH_RATIO04 / 'results/results/classic_tree_shape_sweep.jsonl'
QWEN397_MATH_RATIO04_ENTROPY = QWEN397_MATH_RATIO04 / 'entropy/results/results/classic_tree_shape_sweep.jsonl'
DSPARK = REPO / 'runs/dspark_csd/static_four_methods_four_tasks_20260808_021418'
DSPARK_AIME16_BARE = REPO / 'runs/dspark_csd/aime_pass16_final_20260810'
DSPARK_AIME16_CSD = DSPARK_AIME16_BARE

METHODS = ['Bare', 'Plain CSD', 'Dynamic CSD', 'Dynamic CSD + Entropy Gate']
DISPLAY_METHODS = ['Baseline', 'Static CSD w/o Entropy Gate', 'Dynamic CSD w/o Entropy Gate', 'CSD']
COLORS = ['#A7A7A7', '#9ECAE1', '#4292C6', '#08519C']
DATASETS = ['AIME 2025', 'Math500', 'LiveCodeBench v6', 'GSM8K']

In [ ]:
def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def method_from_tag(tag):
    if tag.startswith('dynamic_entropy_'):
        return 'Dynamic CSD + Entropy Gate'
    if tag.startswith('dynamic_'):
        return 'Dynamic CSD'
    if tag.startswith('plain_'):
        return 'Plain CSD'
    if tag.startswith('eagle_'):
        return 'Bare'
    return None


def extract_qwen_row(row, model, dataset, metric, source):
    metrics = row['other']['metrics']
    tag = row['other']['run_tag']
    method = method_from_tag(tag)
    if method is None:
        return None
    return {
        'model': model, 'dataset': dataset, 'method': method,
        'metric': metric, 'value_percent': 100 * float(metrics[metric]),
        'stderr_percent': 100 * float(metrics.get(metric + '_stderr', 0.0)),
        'source': str(source),
    }


def load_qwen35():
    specs = [
        ('AIME 2025', QWEN35/'gpu4_7/results/classic_tree_shape_sweep.jsonl', 'aime25_avg|0', 'pass@k:k=16&n=16'),
        ('Math500', QWEN35/'gpu4_7/results/classic_tree_shape_sweep.jsonl', 'math_500|0', 'pass@k:k=4&n=4'),
        ('LiveCodeBench v6', QWEN35/'gpu0_3/results/classic_tree_shape_sweep.jsonl', 'lcb:codegeneration_v6|0', 'codegen_pass@4'),
        ('GSM8K', QWEN35/'gsm8k_avg_gpu0_3/results/classic_tree_shape_sweep.jsonl', 'gsm8k_avg|0', 'pass@k:k=4&n=4'),
    ]
    out = []
    for dataset, path, task, metric in specs:
        rows = read_jsonl(path)
        if dataset == 'Math500':
            p30 = QWEN35/'math500_entropy_p30_gpu0_3/results/classic_tree_shape_sweep.jsonl'
            rows += read_jsonl(p30)
        for row in rows:
            if row.get('task') != task:
                continue
            item = extract_qwen_row(row, 'Qwen3.5-35B-A3B', dataset, metric, path)
            if item:
                # Math500 的最终 entropy gate 采用 P30 补跑，排除主文件中的 P20。
                tag = row['other']['run_tag']
                if dataset == 'Math500' and item['method'] == 'Dynamic CSD + Entropy Gate':
                    if not tag.startswith('dynamic_entropy_ignore_ratio_'):
                        continue
                    item['source'] = str(p30)
                out.append(item)
    return out


def load_qwen397():
    specs = [
        ('AIME 2025', 'aime25_avg4', 'aime25_avg|0', 'pass@k:k=16&n=16'),
        ('Math500', 'math500_avg4', 'math_500|0', 'pass@k:k=4&n=4'),
        ('LiveCodeBench v6', 'lcb_avg4', 'lcb:codegeneration_v6|0', 'codegen_pass@4'),
        ('GSM8K', 'gsm8k_avg4', 'gsm8k_avg|0', 'pass@k:k=4&n=4'),
    ]
    out = []
    for dataset, folder, task, metric in specs:
        path = QWEN397/folder/'results/classic_tree_shape_sweep.jsonl'
        paths = [path]
        if dataset == 'Math500':
            paths += [QWEN397_MATH_RATIO04_SD, QWEN397_MATH_RATIO04_ENTROPY]
        for source in paths:
            for row in read_jsonl(source):
                if row.get('task') != task:
                    continue
                item = extract_qwen_row(row, 'Qwen3.5-397B-A17B', dataset, metric, source)
                if not item:
                    continue
                if dataset == 'Math500':
                    selected = {
                        'Bare': path,
                        'Plain CSD': QWEN397_MATH_RATIO04_SD,
                        'Dynamic CSD': QWEN397_MATH_RATIO04_SD,
                        'Dynamic CSD + Entropy Gate': QWEN397_MATH_RATIO04_ENTROPY,
                    }[item['method']]
                    if source != selected:
                        continue
                out.append(item)
    return out


In [ ]:
DSPARK_METHOD_DIR = {
    'Bare': 'bare', 'Plain CSD': 'plain',
    'Dynamic CSD': 'dynamic', 'Dynamic CSD + Entropy Gate': 'entropy',
}


def lighteval_all_metrics(path):
    obj = json.loads(Path(path).read_text(encoding='utf-8'))
    return obj['results']['results']['all']


def dspark_static_path(dataset, method):
    d = DSPARK_METHOD_DIR[method]
    if dataset == 'LiveCodeBench v6':
        return DSPARK/'01_lcb'/d/'lcb_avg4/eval/result.json'
    if dataset == 'AIME 2025':
        return DSPARK/'02_aime_math_gsm8k'/d/'aime25_avg4/eval/result.json'
    if dataset == 'Math500':
        return DSPARK/'02_aime_math_gsm8k'/d/'math500_avg4/eval/result.json'
    # GSM8K bare 在第二阶段，其余三种方法在补齐阶段。
    stage = '02_aime_math_gsm8k' if method == 'Bare' else '03_gsm8k_remaining'
    return DSPARK/stage/d/'gsm8k_avg4/eval/result.json'


def dspark_aime16_path(method):
    root = DSPARK_AIME16_BARE if method == 'Bare' else DSPARK_AIME16_CSD
    return root/DSPARK_METHOD_DIR[method]/'aime25_avg16/eval/result.json'

def dspark_aime16_complete():
    return all(dspark_aime16_path(method).is_file() for method in DSPARK_METHOD_DIR)


def load_dspark():
    use_aime16 = dspark_aime16_complete()
    out = []
    for dataset in DATASETS:
        for method, d in DSPARK_METHOD_DIR.items():
            if dataset == 'AIME 2025' and use_aime16:
                path = dspark_aime16_path(method)
                metric = 'pass@k:k=16&n=16'
            else:
                path = dspark_static_path(dataset, method)
                metric = {
                    'AIME 2025': 'pass@k:k=4&n=4',
                    'Math500': 'pass@k:k=4&n=4',
                    'LiveCodeBench v6': 'codegen_pass@4',
                    'GSM8K': 'pass@k:k=4&n=4',
                }[dataset]
            metrics = lighteval_all_metrics(path)
            out.append({
                'model': 'DeepSeek-V4-Flash-DSpark', 'dataset': dataset,
                'method': method, 'metric': metric,
                'value_percent': 100 * float(metrics[metric]),
                'stderr_percent': 100 * float(metrics.get(metric + '_stderr', 0.0)),
                'source': str(path),
            })
    return out


records = load_qwen35() + load_qwen397() + load_dspark()
df = pd.DataFrame(records)
df['model'] = pd.Categorical(df['model'], [
    'Qwen3.5-35B-A3B', 'Qwen3.5-397B-A17B', 'DeepSeek-V4-Flash-DSpark'
])
df['dataset'] = pd.Categorical(df['dataset'], DATASETS)
df['method'] = pd.Categorical(df['method'], METHODS)
df = df.sort_values(['model', 'dataset', 'method']).reset_index(drop=True)

expected = 3 * len(DATASETS) * len(METHODS)
assert len(df) == expected, f'Expected {expected} rows, got {len(df)}'
assert not df.duplicated(['model', 'dataset', 'method']).any()
df.to_csv(OUT_DIR/'three_model_pass_at_n_data.csv', index=False)
print('DSpark AIME metric:', 'Pass@16' if dspark_aime16_complete() else 'Pass@4 fallback')
df

In [ ]:
summary = df.pivot_table(
    index=['model', 'dataset'], columns='method', values='value_percent', observed=True
).round(2)
summary

In [ ]:
plt.rcParams.update({
    'font.size': 13, 'axes.titlesize': 18, 'axes.labelsize': 15,
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 16,
})

models = ['Qwen3.5-35B-A3B', 'Qwen3.5-397B-A17B', 'DeepSeek-V4-Flash-DSpark']
fig, axes = plt.subplots(1, 3, figsize=(25, 7.2), sharey=True)
x = np.arange(len(DATASETS))
width = 0.22

for ax, model in zip(axes, models):
    panel = df[df['model'] == model]
    for mi, (method, color) in enumerate(zip(METHODS, COLORS)):
        rows = panel[panel['method'] == method].set_index('dataset').reindex(DATASETS)
        values = rows['value_percent'].to_numpy(float)
        positions = x + (mi - 1.5) * width
        bars = ax.bar(positions, values, width, label=method, color=color,
                      edgecolor='white', linewidth=0.8)
        for bar, value in zip(bars, values):
            text_color = 'white' if mi >= 2 else '#222222'
            ax.text(bar.get_x() + bar.get_width()/2, value + 0.28, f'{value:.1f}',
                    ha='center', va='bottom', fontsize=9.0, rotation=0,
                    color='#063D78' if mi == 3 else '#222222',
                    fontweight='bold' if mi == 3 else 'normal', zorder=6)
    ax.set_title(model, pad=16)
    ax.set_xticks(x)
    labels = ['AIME\n' + ('Pass@16' if model != 'DeepSeek-V4-Flash-DSpark' or dspark_aime16_complete() else 'Pass@4'),
              'Math500\nPass@4', 'LCB v6\nPass@4', 'GSM8K\nPass@4']
    ax.set_xticklabels(labels)
    ax.grid(axis='y', linestyle='--', linewidth=0.9, alpha=0.28)
    ax.set_axisbelow(True)
    ax.set_ylim(75, 103.0)

axes[0].set_ylabel('Pass@n (%)')
handles, _ = axes[0].get_legend_handles_labels()
fig.legend(handles, DISPLAY_METHODS, loc='upper center', bbox_to_anchor=(0.5, 1.01),
           ncol=4, frameon=False, fontsize=16, handlelength=1.8, columnspacing=1.8)
fig.tight_layout(rect=[0, 0, 1, 0.90])

png = OUT_DIR/'three_model_pass_at_n.png'
pdf = OUT_DIR/'three_model_pass_at_n.pdf'
fig.savefig(png, dpi=240, bbox_inches='tight')
fig.savefig(pdf, bbox_inches='tight')
plt.show()
print(png)
print(pdf)

## 吞吐与投机成功率

下面两张图直接从上述同一批 run 提取数据。吞吐采用端到端 `output_token_throughput`；投机成功率采用 `aggregate_spec_accept_rate` / `spec_accept_rate`，即正确 draft token 数除以提议 draft token 数。

> 吞吐只适合在同一子图内比较：35B 使用 4×H20，397B 和 DSpark 使用 8×H20。MTP 与 DSpark 的 draft 形状也不同，因此投机成功率不宜跨子图直接对比。

In [ ]:
TASK_BY_DATASET = {
    'AIME 2025': 'aime25_avg|0', 'Math500': 'math_500|0',
    'LiveCodeBench v6': 'lcb:codegeneration_v6|0', 'GSM8K': 'gsm8k_avg|0',
}

def qwen_performance(record):
    path = Path(record['source'])
    task = TASK_BY_DATASET[record['dataset']]
    candidates = []
    for row in read_jsonl(path):
        if row.get('task') == task and method_from_tag(row['other']['run_tag']) == record['method']:
            candidates.append(row)
    assert len(candidates) == 1, (path, task, record['method'], len(candidates))
    row = candidates[0]
    return float(row['throughput']), 100 * float(row['aggregate_spec_accept_rate'])

def dspark_performance(record):
    result_path = Path(record['source'])
    result = json.loads(result_path.read_text(encoding='utf-8'))
    metric_path = result_path.parent.parent/'metrics/task_metrics.json'
    metrics = json.loads(metric_path.read_text(encoding='utf-8'))
    return (float(result['request_timing']['output_token_throughput']),
            100 * float(metrics['spec_accept_rate']))

performance_records = []
for record in df.to_dict('records'):
    throughput, success_rate = (
        dspark_performance(record) if record['model'] == 'DeepSeek-V4-Flash-DSpark'
        else qwen_performance(record)
    )
    performance_records.append({**record, 'throughput_tok_s': throughput,
                                'spec_success_rate_percent': success_rate})

perf_df = pd.DataFrame(performance_records)
perf_df.to_csv(OUT_DIR/'three_model_performance_data.csv', index=False)
perf_df[['model', 'dataset', 'method', 'throughput_tok_s', 'spec_success_rate_percent']]

In [ ]:
THROUGHPUT_PANEL_TITLES = {
    'Qwen3.5-35B-A3B': 'Qwen3.5-35B-A3B\n(TP=4, Concurrency=48, 4×H20)',
    'Qwen3.5-397B-A17B': 'Qwen3.5-397B-A17B\n(TP=8, Concurrency=48, 8×H20)',
    'DeepSeek-V4-Flash-DSpark': 'DeepSeek-V4-Flash-DSpark\n(TP=8, DP=8, Concurrency=64, 8×H20, Static)',
}
SUCCESS_PANEL_TITLES = {
    'Qwen3.5-35B-A3B': 'Qwen3.5-35B-A3B\n(MTP: steps=3, topk=1, draft=4)',
    'Qwen3.5-397B-A17B': 'Qwen3.5-397B-A17B\n(MTP: steps=3, topk=1, draft=4)',
    'DeepSeek-V4-Flash-DSpark': 'DeepSeek-V4-Flash-DSpark\n(DSpark Static: draft=6)',
}

def plot_performance(value_col, ylabel, panel_titles, stem, value_format, sharey, delta_mode):
    fig, axes = plt.subplots(1, 3, figsize=(25, 7.2), sharey=sharey)
    x = np.arange(len(DATASETS)); width = 0.22
    for ax, model in zip(axes, models):
        panel = perf_df[perf_df['model'] == model]
        panel_max = panel[value_col].max()
        baseline = panel[panel['method'] == 'Bare'].set_index('dataset').reindex(DATASETS)[value_col].to_numpy(float)
        for mi, (method, color) in enumerate(zip(METHODS, COLORS)):
            rows = panel[panel['method'] == method].set_index('dataset').reindex(DATASETS)
            values = rows[value_col].to_numpy(float)
            is_final = method == 'Dynamic CSD + Entropy Gate'
            bars = ax.bar(x + (mi - 1.5) * width, values, width, label=method, color=color,
                          edgecolor='#063D78' if is_final else 'white',
                          linewidth=1.6 if is_final else 0.8, zorder=3 if is_final else 2)
            for di, (bar, value) in enumerate(zip(bars, values)):
                cx = bar.get_x() + bar.get_width()/2
                if delta_mode == 'percent':
                    # 吞吐图只标注相对 bare 的加速比，不叠加绝对吞吐数值。
                    ax.text(cx, value + panel_max * 0.012, f'{value / baseline[di]:.2f}×',
                            ha='center', va='bottom', fontsize=9.0,
                            color='#063D78' if is_final else '#222222',
                            fontweight='bold' if is_final else 'normal', zorder=6)
                else:
                    ax.text(cx, value + panel_max * 0.010, value_format.format(value),
                            ha='center', va='bottom', fontsize=9.0, rotation=0,
                            color='#063D78' if is_final else '#222222',
                            fontweight='bold' if is_final else 'normal')
        ax.set_title(panel_titles[model], pad=16)
        ax.set_xticks(x); ax.set_xticklabels(['AIME', 'Math500', 'LCB v6', 'GSM8K'])
        ax.grid(axis='y', linestyle='--', linewidth=0.9, alpha=0.28); ax.set_axisbelow(True)
        if not sharey: ax.set_ylim(0, panel_max * 1.22)
    if sharey: axes[0].set_ylim(0, 100)
    axes[0].set_ylabel(ylabel)
    handles, _ = axes[0].get_legend_handles_labels()
    fig.legend(handles, DISPLAY_METHODS, loc='upper center', bbox_to_anchor=(0.5, 1.01),
               ncol=4, frameon=False, fontsize=16, handlelength=1.8, columnspacing=1.8)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    png, pdf = OUT_DIR/f'{stem}.png', OUT_DIR/f'{stem}.pdf'
    fig.savefig(png, dpi=240, bbox_inches='tight'); fig.savefig(pdf, bbox_inches='tight')
    plt.show(); print(png); print(pdf)
    return fig


In [ ]:
throughput_fig = plot_performance(
    'throughput_tok_s', 'Output throughput (token/s)', THROUGHPUT_PANEL_TITLES,
    'three_model_throughput', '{:.0f}', sharey=False, delta_mode='percent',
)

In [ ]:
success_rate_fig = plot_performance(
    'spec_success_rate_percent', 'Draft-token success rate (%)', SUCCESS_PANEL_TITLES,
    'three_model_spec_success_rate', '{:.1f}', sharey=True, delta_mode='pp',
)